# Scripted F-16 manoeuvre video with BVR Sim / JSBSim

This notebook runs an **open-loop demonstration** through the same JSBSim flight-dynamics engine used by BVR Sim, records the F-16 state, and renders an MP4 (or GIF fallback). The script commands a level departure, climbing left turn, reversal, and recovery.

> **Scope:** synthetic simulation only. The control schedule is deliberately simple and is not flight-control or operational guidance.

The companion [setup and troubleshooting guide](../docs/f16_video_guide.md) explains how to pin BVR Sim, locate its aircraft data, install FFmpeg, and use BVR Sim's ACMI output when a Tacview view is preferred.


## 1. Install prerequisites

Create a clean environment from the repository root. Uncomment the `%pip` line if this kernel does not already have the video dependencies. Restart the kernel after installation.


In [ ]:
# %pip install -e ".[video]"

from pathlib import Path
import os

import jsbsim
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import pandas as pd
from IPython.display import Video, Image, display

OUTPUT_DIR = Path("artifacts/f16_video")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("JSBSim", getattr(jsbsim, "__version__", "unknown"))


## 2. Configure the F-16 and manoeuvre

`JSBSIM_ROOT` is optional. Leave it unset to use the aircraft bundled with the `jsbsim` wheel. To use the exact aircraft definitions from a BVR Sim checkout, point it at the directory containing `aircraft/`, `engine/`, and `systems/`.

The initial conditions use JSBSim's imperial property interface. The saved table also contains SI units for downstream use.


In [ ]:
JSBSIM_ROOT = os.getenv("JSBSIM_ROOT") or None
MODEL = os.getenv("JSBSIM_MODEL", "f16")
DT = 1 / 60  # simulation step; video is sampled separately
VIDEO_FPS = 30
PLAYBACK_SPEED = 2.0
SEED = 7  # recorded for reproducibility; this schedule itself is deterministic

INITIAL = {
    "ic/lat-gc-deg": 37.62,
    "ic/long-gc-deg": -122.38,
    "ic/h-sl-ft": 15_000.0,
    "ic/psi-true-deg": 90.0,
    "ic/u-fps": 750.0,
    "ic/v-fps": 0.0,
    "ic/w-fps": 0.0,
    "ic/p-rad_sec": 0.0,
    "ic/q-rad_sec": 0.0,
    "ic/r-rad_sec": 0.0,
}

# (end time, phase label, aileron, elevator, rudder, throttle), all controls normalized.
MANOEUVRE = [
    (5.0,  "level departure",  0.00,  0.00, 0.00, 0.78),
    (13.0, "climbing left turn", -0.28, -0.10, 0.00, 0.88),
    (17.0, "unload",            0.12,  0.04, 0.00, 0.82),
    (26.0, "right reversal",    0.32,  0.06, 0.00, 0.84),
    (34.0, "recover level",    -0.10, -0.02, 0.00, 0.76),
    (40.0, "level exit",        0.00,  0.00, 0.00, 0.74),
]


## 3. Run JSBSim and record the trajectory

If `load_model` fails, follow the model-path checks in the guide. Property names below are standard JSBSim catalog names; the helper raises a clear error if the selected F-16 definition does not expose one.


In [ ]:
def require_property(fdm, name):
    try:
        return float(fdm[name])
    except Exception as exc:
        raise RuntimeError(f"The model does not expose required JSBSim property {name!r}") from exc


def active_phase(time_s):
    return next(row for row in MANOEUVRE if time_s < row[0])


def run_manoeuvre():
    fdm = jsbsim.FGFDMExec(JSBSIM_ROOT)
    fdm.set_debug_level(0)
    fdm.set_dt(DT)
    if not fdm.load_model(MODEL):
        raise RuntimeError(
            f"Could not load {MODEL!r}. Set JSBSIM_ROOT to a directory containing "
            "aircraft/, engine/, and systems/."
        )
    for name, value in INITIAL.items():
        fdm[name] = value
    if not fdm.run_ic():
        raise RuntimeError("JSBSim rejected the initial conditions")

    rows = []
    while fdm.get_sim_time() < MANOEUVRE[-1][0]:
        time_s = fdm.get_sim_time()
        _, phase, aileron, elevator, rudder, throttle = active_phase(time_s)
        fdm["fcs/aileron-cmd-norm"] = aileron
        fdm["fcs/elevator-cmd-norm"] = elevator
        fdm["fcs/rudder-cmd-norm"] = rudder
        fdm["fcs/throttle-cmd-norm"] = throttle
        if not fdm.run():
            raise RuntimeError(f"JSBSim stopped at t={time_s:.2f} s")
        rows.append({
            "time_s": fdm.get_sim_time(), "phase": phase,
            "latitude_deg": require_property(fdm, "position/lat-gc-deg"),
            "longitude_deg": require_property(fdm, "position/long-gc-deg"),
            "altitude_m": require_property(fdm, "position/h-sl-ft") * 0.3048,
            "roll_deg": require_property(fdm, "attitude/phi-deg"),
            "pitch_deg": require_property(fdm, "attitude/theta-deg"),
            "heading_deg": require_property(fdm, "attitude/psi-deg"),
            "speed_mps": require_property(fdm, "velocities/vtrue-fps") * 0.3048,
        })
    return pd.DataFrame(rows)

trajectory = run_manoeuvre()
trajectory.to_csv(OUTPUT_DIR / "f16_scripted_manoeuvre.csv", index=False)
trajectory.tail()


## 4. Sanity-check the run

Review the plots before rendering. Extreme attitudes, rapid divergence, or an implausible speed trace indicate that the aircraft definition expects a different controller/autopilot path; see the guide rather than tuning blindly.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for axis, column, label in zip(
    axes.flat,
    ["altitude_m", "speed_mps", "roll_deg", "pitch_deg"],
    ["Altitude (m)", "True speed (m/s)", "Roll (deg)", "Pitch (deg)"],
):
    axis.plot(trajectory.time_s, trajectory[column])
    axis.set_ylabel(label)
    axis.grid(alpha=0.25)
for axis in axes[-1]:
    axis.set_xlabel("Simulation time (s)")
fig.suptitle("F-16 scripted manoeuvre checks")
fig.tight_layout()


## 5. Render the video

The camera uses a local tangent-plane approximation, appropriate for this short demonstration. MP4 output requires an FFmpeg executable visible to Matplotlib. When it is unavailable, the cell writes a GIF with Pillow instead.


In [ ]:
def add_local_coordinates(frame):
    lat0 = np.deg2rad(frame.latitude_deg.iloc[0])
    frame = frame.copy()
    frame["east_m"] = np.deg2rad(frame.longitude_deg - frame.longitude_deg.iloc[0]) * 6_378_137 * np.cos(lat0)
    frame["north_m"] = np.deg2rad(frame.latitude_deg - frame.latitude_deg.iloc[0]) * 6_378_137
    frame["up_m"] = frame.altitude_m - frame.altitude_m.iloc[0]
    return frame


def render_video(frame):
    frame = add_local_coordinates(frame)
    stride = max(1, round(PLAYBACK_SPEED / (DT * VIDEO_FPS)))
    frames = frame.iloc[::stride].reset_index(drop=True)
    fig = plt.figure(figsize=(10, 7), dpi=120)
    axis = fig.add_subplot(111, projection="3d")
    axis.set(xlabel="East (m)", ylabel="North (m)", zlabel="Height above start (m)")
    axis.set_title("JSBSim F-16 — scripted manoeuvre")
    axis.set_xlim(frames.east_m.min() - 100, frames.east_m.max() + 100)
    axis.set_ylim(frames.north_m.min() - 100, frames.north_m.max() + 100)
    axis.set_zlim(frames.up_m.min() - 100, frames.up_m.max() + 100)
    trail, = axis.plot([], [], [], color="#2b6cb0", linewidth=2)
    aircraft, = axis.plot([], [], [], marker=">", color="#d53f3f", markersize=9)
    status = axis.text2D(0.02, 0.96, "", transform=axis.transAxes)

    def update(index):
        current = frames.iloc[index]
        trail.set_data_3d(frames.east_m[:index + 1], frames.north_m[:index + 1], frames.up_m[:index + 1])
        aircraft.set_data_3d([current.east_m], [current.north_m], [current.up_m])
        status.set_text(
            f"t={current.time_s:5.1f} s | {current.phase}\n"
            f"roll={current.roll_deg:5.1f}°  pitch={current.pitch_deg:5.1f}°  "
            f"speed={current.speed_mps:5.0f} m/s"
        )
        return trail, aircraft, status

    movie = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / VIDEO_FPS, blit=False)
    if animation.writers.is_available("ffmpeg"):
        path = OUTPUT_DIR / "f16_scripted_manoeuvre.mp4"
        movie.save(path, writer=animation.FFMpegWriter(fps=VIDEO_FPS, bitrate=2400), dpi=120)
    else:
        path = OUTPUT_DIR / "f16_scripted_manoeuvre.gif"
        movie.save(path, writer=animation.PillowWriter(fps=VIDEO_FPS), dpi=90)
    plt.close(fig)
    return path

video_path = render_video(trajectory)
print(f"Wrote {video_path.resolve()}")
if video_path.suffix == ".mp4":
    display(Video(str(video_path), embed=True))
else:
    display(Image(filename=str(video_path)))


## 6. Reusing the result with BVR Sim

- Keep the BVR Sim revision, JSBSim version, model name, initial conditions, `DT`, and `MANOEUVRE` beside every output.
- For dataset generation, translate each phase to BVR Sim's native `MultiDiscrete([15, 15, 9, 2])` action or a BVR scripted-policy class; do **not** send normalized FCS values to the BVR environment action API.
- For a combat-replay view, enable BVR Sim's ACMI/Tacview logger and open the `.acmi` output in Tacview. The MP4 above is a portable diagnostic view and does not claim to be BVR Sim's native renderer.
- Before a long batch, compare a short BVR rollout with this JSBSim run and archive the resolved configuration. See the companion guide for a checklist.
